In [1]:
from pyspark.sql import functions as F

# Bronze-Daten laden
df_bronze = spark.table(
    "bronze.synthetic_vehicle_diagnostics"
)

# Datensätze vor Bereinigung
bronze_count = df_bronze.count()

print("Bronze-Datensätze:", bronze_count)

# Pflichtfelder und Status normalisieren
df_prepared = (
    df_bronze
    .withColumn("ECU", F.trim(F.col("ECU")))
    .withColumn("DTC", F.upper(F.trim(F.col("DTC"))))
    .withColumn(
        "ErrorCategory",
        F.trim(F.col("ErrorCategory"))
    )
    .withColumn(
        "TestResult",
        F.upper(F.trim(F.col("TestResult")))
    )
    .withColumn(
        "TestTimestamp",
        F.col("TestTimestamp").cast("timestamp")
    )
)

# Zulässige Fehlerkategorien
valid_categories = [
    "Short_Circuit",
    "Overtemperature",
    "Communication_Loss",
    "Voltage_Deviation",
    "Sensor_Failure",
    "Diagnostic_Error"
]

# Qualitätsregeln
valid_condition = (
    F.col("TestID").isNotNull()
    & F.col("ECU").isNotNull()
    & (F.length(F.col("ECU")) > 0)
    & F.col("DTC").rlike("^[PCBU][0-9A-F]{4}$")
    & F.col("ErrorCategory").isin(valid_categories)
    & F.col("TestResult").isin("PASS", "FAIL")
    & F.col("TestTimestamp").isNotNull()
)

# Ungültige Datensätze ermitteln
invalid_count = df_prepared.filter(
    ~valid_condition
).count()

# Gültige Datensätze auswählen
df_valid = df_prepared.filter(valid_condition)

# Duplikate entfernen
valid_before_dedup = df_valid.count()

df_silver_advanced = df_valid.dropDuplicates(
    ["TestID"]
)

silver_count = df_silver_advanced.count()

duplicate_removed = valid_before_dedup - silver_count

# Validierungsbericht
print("Bronze:", bronze_count)
print("Ungültige Datensätze:", invalid_count)
print("Entfernte Duplikate:", duplicate_removed)
print("Silver:", silver_count)

display(df_silver_advanced.limit(10))



StatementMeta(, f2f6bc77-1a00-4a93-b5ee-10754b8b3102, 3, Finished, Available, Finished, False)

Bronze-Datensätze: 1000
Bronze: 1000
Ungültige Datensätze: 0
Entfernte Duplikate: 0
Silver: 1000


SynapseWidget(Synapse.DataFrame, d334af42-54a5-42f0-a4c2-05daea28b451)

In [2]:
# Silver-Daten dauerhaft als Delta-Tabelle speichern

silver_table = "silver.synthetic_vehicle_diagnostics_clean"

(
    df_silver_advanced.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print("Silver-Tabelle erfolgreich gespeichert!")
print("Tabellenname:", silver_table)


StatementMeta(, f2f6bc77-1a00-4a93-b5ee-10754b8b3102, 4, Finished, Available, Finished, False)

Silver-Tabelle erfolgreich gespeichert!
Tabellenname: silver.synthetic_vehicle_diagnostics_clean


In [3]:
# Silver-Tabelle erneut aus dem Lakehouse lesen

df_check = spark.table(
    "silver.synthetic_vehicle_diagnostics_clean"
)

print("Gespeicherte Datensätze:", df_check.count())

display(df_check.limit(10))


StatementMeta(, f2f6bc77-1a00-4a93-b5ee-10754b8b3102, 5, Finished, Available, Finished, False)

Gespeicherte Datensätze: 1000


SynapseWidget(Synapse.DataFrame, 74810049-c9d5-409c-a866-63b155611ea0)